In [ ]:
storage_account = 'adlsretailpulse<initials>'
container       = 'retaildata'

df_raw = spark.read \
    .option('header', True) \
    .option('inferSchema', True) \
    .csv(f'abfss://{container}@{storage_account}.dfs.core.windows.net/raw/products/')

print(f'Row count: {df_raw.count():,}')   # Expected: 500
df_raw.printSchema()
df_raw.show(10)

StatementMeta(sparkRetail, 6, 6, Finished, Available, Finished, False)

Row count: 500
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- cost_price: double (nullable = true)
 |-- supplier: string (nullable = true)

+----------+------------+--------+------------+----------+-----------+
|product_id|product_name|category|sub_category|cost_price|   supplier|
+----------+------------+--------+------------+----------+-----------+
|         1|   Product_1|    Home|       Decor|    218.87|Supplier_19|
|         2|   Product_2| Grocery|   Beverages|    340.22| Supplier_2|
|         3|   Product_3| Grocery|       Dairy|    153.12|Supplier_17|
|         4|   Product_4|    Home|     Kitchen|    258.39|Supplier_18|
|         5|   Product_5| Grocery|      Bakery|    375.63| Supplier_4|
|         6|   Product_6|Clothing|       Women|    213.82|Supplier_17|
|         7|   Product_7| Grocery|       Dairy|     95.26| Supplier_9|
|         8|   P

In [6]:
from pyspark.sql.functions import col, sum as _sum, min as _min, max as _max

# Check nulls
null_counts = df_raw.select([
    _sum(col(c).isNull().cast('int')).alias(c)
    for c in df_raw.columns
])
null_counts.show()
# Expected: all zeros — products file is clean

# Check categories and sub-categories
print('Categories:')
df_raw.groupBy('category').count().show()
# Expected: Home, Grocery, Clothing, Electronics

print('Sub-categories per category:')
df_raw.groupBy('category', 'sub_category').count().orderBy('category').show(20)

# Cost price range
df_raw.select(
    _min('cost_price').alias('min_price'),
    _max('cost_price').alias('max_price')
).show()
# Expected: min ~2.02, max ~397.90

# Check for duplicate product_ids
from pyspark.sql.functions import count
dupes = df_raw.groupBy('product_id').count().filter(col('count') > 1)
print(f'Duplicate product_ids: {dupes.count()}')
# Expected: 0

StatementMeta(sparkRetail, 6, 7, Finished, Available, Finished, False)

+----------+------------+--------+------------+----------+--------+
|product_id|product_name|category|sub_category|cost_price|supplier|
+----------+------------+--------+------------+----------+--------+
|         0|           0|       0|           0|         0|       0|
+----------+------------+--------+------------+----------+--------+

Categories:
+-----------+-----+
|   category|count|
+-----------+-----+
|       Home|  124|
|    Grocery|  128|
|Electronics|  130|
|   Clothing|  118|
+-----------+-----+

Sub-categories per category:
+-----------+------------+-----+
|   category|sub_category|count|
+-----------+------------+-----+
|   Clothing|         Men|   42|
|   Clothing|       Women|   42|
|   Clothing|        Kids|   34|
|Electronics|     Laptops|   38|
|Electronics| Accessories|   44|
|Electronics|      Phones|   48|
|    Grocery|      Bakery|   43|
|    Grocery|   Beverages|   47|
|    Grocery|       Dairy|   38|
|       Home|     Kitchen|   37|
|       Home|   Furniture|  

In [7]:
from pyspark.sql.functions import trim, initcap, col, round as _round

df_clean = (df_raw
    # Correct types
    .withColumn('product_id',  col('product_id').cast('int'))
    .withColumn('cost_price',  col('cost_price').cast('double'))

    # Standardise text — trim whitespace, fix casing
    .withColumn('product_name',  trim(col('product_name')))
    .withColumn('category',      trim(initcap(col('category'))))
    .withColumn('sub_category',  trim(initcap(col('sub_category'))))
    .withColumn('supplier',      trim(col('supplier')))

    # Filter out invalid prices (cost must be > 0)
    .filter(col('cost_price') > 0)

    # Drop duplicates on product_id
    .dropDuplicates(['product_id'])
)

print(f'Clean row count: {df_clean.count():,}')
# Expected: 500
df_clean.show(10)

StatementMeta(sparkRetail, 6, 8, Finished, Available, Finished, False)

Clean row count: 500
+----------+------------+--------+------------+----------+-----------+
|product_id|product_name|category|sub_category|cost_price|   supplier|
+----------+------------+--------+------------+----------+-----------+
|         1|   Product_1|    Home|       Decor|    218.87|Supplier_19|
|         2|   Product_2| Grocery|   Beverages|    340.22| Supplier_2|
|         3|   Product_3| Grocery|       Dairy|    153.12|Supplier_17|
|         4|   Product_4|    Home|     Kitchen|    258.39|Supplier_18|
|         5|   Product_5| Grocery|      Bakery|    375.63| Supplier_4|
|         6|   Product_6|Clothing|       Women|    213.82|Supplier_17|
|         7|   Product_7| Grocery|       Dairy|     95.26| Supplier_9|
|         8|   Product_8| Grocery|   Beverages|    117.83|Supplier_11|
|         9|   Product_9| Grocery|   Beverages|     35.57| Supplier_6|
|        10|  Product_10|Clothing|         Men|     355.0|Supplier_19|
+----------+------------+--------+------------+---------

In [8]:
from pyspark.sql.functions import when, col, round as _round

df_enriched = (df_clean
    # Price tier — useful for filtering in analytics
    .withColumn('price_tier',
        when(col('cost_price') < 50,  'Budget')
       .when(col('cost_price') < 150, 'Mid-Range')
       .when(col('cost_price') < 300, 'Premium')
       .otherwise('Luxury'))

    # Suggested retail price — standard 40% markup over cost
    .withColumn('suggested_retail_price',
        _round(col('cost_price') * 1.40, 2))

    # Gross margin percentage — how much profit margin at retail
    .withColumn('margin_pct',
        _round(((col('suggested_retail_price') - col('cost_price'))
                / col('suggested_retail_price')) * 100, 1))
)

print('Price tier distribution:')
df_enriched.groupBy('price_tier').count().orderBy('price_tier').show()

print('Category averages:')
df_enriched.groupBy('category') \
    .agg(
        _round(df_enriched['cost_price'].cast('double')
               .__class__.__name__ and
               __import__('pyspark.sql.functions', fromlist=['avg'])
               .avg('cost_price'), 2).alias('avg_cost'),
    ).show()

StatementMeta(sparkRetail, 6, 9, Finished, Available, Finished, False)

Price tier distribution:
+----------+-----+
|price_tier|count|
+----------+-----+
|    Budget|   79|
|    Luxury|  113|
| Mid-Range|  123|
|   Premium|  185|
+----------+-----+

Category averages:
+-----------+--------+
|   category|avg_cost|
+-----------+--------+
|       Home|  200.64|
|    Grocery|  212.88|
|Electronics|  175.33|
|   Clothing|  179.38|
+-----------+--------+



In [9]:
from pyspark.sql.functions import avg, round as _round, when, col

df_enriched = (df_clean
    .withColumn('price_tier',
        when(col('cost_price') < 50,  'Budget')
       .when(col('cost_price') < 150, 'Mid-Range')
       .when(col('cost_price') < 300, 'Premium')
       .otherwise('Luxury'))
    .withColumn('suggested_retail_price',
        _round(col('cost_price') * 1.40, 2))
    .withColumn('margin_pct',
        _round(((col('suggested_retail_price') - col('cost_price'))
                / col('suggested_retail_price')) * 100, 1))
)

# Verify distributions
df_enriched.groupBy('price_tier').count().orderBy('price_tier').show()
df_enriched.groupBy('category') \
    .agg(_round(avg('cost_price'), 2).alias('avg_cost_price')) \
    .orderBy('category').show()

StatementMeta(sparkRetail, 6, 10, Finished, Available, Finished, False)

+----------+-----+
|price_tier|count|
+----------+-----+
|    Budget|   79|
|    Luxury|  113|
| Mid-Range|  123|
|   Premium|  185|
+----------+-----+

+-----------+--------------+
|   category|avg_cost_price|
+-----------+--------------+
|   Clothing|        179.38|
|Electronics|        175.33|
|    Grocery|        212.88|
|       Home|        200.64|
+-----------+--------------+



In [10]:
silver_path = (
    f'abfss://{container}@{storage_account}'
    f'.dfs.core.windows.net/curated/products/'
)

df_enriched.write \
    .mode('overwrite') \
    .parquet(silver_path)

print('✅ Products Silver layer written successfully!')

# Verify
df_verify = spark.read.parquet(silver_path)
print(f'Verified row count: {df_verify.count():,}')
# Expected: 500
df_verify.printSchema()
df_verify.show(5)

StatementMeta(sparkRetail, 6, 11, Finished, Available, Finished, False)

✅ Products Silver layer written successfully!
Verified row count: 500
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- cost_price: double (nullable = true)
 |-- supplier: string (nullable = true)
 |-- price_tier: string (nullable = true)
 |-- suggested_retail_price: double (nullable = true)
 |-- margin_pct: double (nullable = true)

+----------+------------+--------+------------+----------+-----------+----------+----------------------+----------+
|product_id|product_name|category|sub_category|cost_price|   supplier|price_tier|suggested_retail_price|margin_pct|
+----------+------------+--------+------------+----------+-----------+----------+----------------------+----------+
|         1|   Product_1|    Home|       Decor|    218.87|Supplier_19|   Premium|                306.42|      28.6|
|         2|   Product_2| Grocery|   Beverages|    340.22| Supplier_2